# Sensitive analysis

In [1]:
import pandas as pd
import numpy as np
import pickle

from thesis.correlation_sociodemographic_covid.util import save_model, summarize_results, calculate_95_ci

## Loading data

In [2]:
df_deaths = pd.read_csv('data/df_mortality.csv', index_col=0)
df_labeled_cluster = pd.read_csv('data/df_labeled_cluster.csv', index_col=0)
df_deaths['cluster_label'] = df_labeled_cluster['cluster_label']

## Variable Sensitivity Analysis

In [3]:
df_difference_statistics = pd.DataFrame()
df_difference_coefficients = pd.DataFrame()

list_periods = ['2020_1','2020', '2021', '2022', '2020_2022']

for model_id in [11]:
    for i in range(5):
        period = list_periods[i]
        print('\n*** Period: ', period)
 
        print('\*** Model',model_id)
        print('===>Full model:')
        with open('models/model_'+str(model_id)+'_'+period+'.pkl', 'rb') as file:
            model_reference = pickle.load(file)
        # summarize_results(model_reference)
        model_reference_r2_cs = model_reference.pseudo_rsquared()
                
        for removed_variable in model_reference.params[1:].index:
            print('\n*** Removed variable: ', removed_variable)        
            with open('models/sensitivity_analysis/variable/model_' + str(model_id) +'_' + period +'_' + removed_variable + '.pkl', 'rb') as file:
                model_variable = pickle.load(file)        
            # summarize_results(model_variable)
            model_variable_r2 = model_variable.pseudo_rsquared()
            
            r2_cs_difference = model_variable_r2 - model_reference_r2_cs
            statistic_row = {'model_id':model_id, 'period': period, 'variable':removed_variable, 'r2_difference':r2_cs_difference}
            new_row = pd.DataFrame([statistic_row])
            df_difference_statistics = pd.concat([df_difference_statistics, new_row], ignore_index=True)
            
            for variable in model_variable.params.index:
                coefficient_difference = model_variable.params[variable] - model_reference.params[variable]
                coefficient_relative_difference = coefficient_difference / model_reference.params[variable]
                coefficient_row = {'model_id': model_id, 'period': period, 'removed_variable': removed_variable, 'variable': variable, 'coefficient_difference': coefficient_difference, 'coefficient_relative_difference': coefficient_relative_difference}
                new_row = pd.DataFrame([coefficient_row])
                df_difference_coefficients = pd.concat([df_difference_coefficients, new_row], ignore_index=True)


*** Period:  2020_1
\*** Model 11
===>Full model:

*** Removed variable:  percentage_urban_population

*** Removed variable:  percentage_indigenous_population

*** Removed variable:  density_median_effectively_domiciled_area

*** Removed variable:  gini

*** Removed variable:  percentage_estimated_households_in_informal_settlements

*** Removed variable:  demographic_density_in_informal_settlements

*** Removed variable:  percentage_hospitalizations_diseases_inadequate_sanitation

*** Removed variable:  percentage_self_employed_workers

*** Removed variable:  unemployment_rate

*** Removed variable:  percentage_workers_commerce

*** Removed variable:  percentage_workers_services

*** Removed variable:  percentage_workers_industry

*** Removed variable:  expected_years_of_schooling_at_age_18

*** Removed variable:  percentage_votes_for_bolsonaro

*** Period:  2020
\*** Model 11
===>Full model:

*** Removed variable:  percentage_urban_population

*** Removed variable:  percentage_indige

In [4]:
df_difference_statistics['r2_difference_absolute'] = df_difference_statistics['r2_difference'].abs()
df_difference_statistics.sort_values(['model_id', 'period', 'r2_difference'], ascending=[True, True, True])[['model_id', 'period', 'variable', 'r2_difference']].round(3)

,model_id,period,variable,r2_difference
16,11,2020,density_median_effectively_domiciled_area,-0.026
15,11,2020,percentage_indigenous_population,-0.011
27,11,2020,percentage_votes_for_bolsonaro,-0.011
26,11,2020,expected_years_of_schooling_at_age_18,-0.008
24,11,2020,percentage_workers_services,-0.006
...,...,...,...,...
52,11,2022,percentage_workers_services,-0.001
46,11,2022,percentage_estimated_households_in_informal_se...,-0.000
43,11,2022,percentage_indigenous_population,-0.000
50,11,2022,unemployment_rate,-0.000


In [5]:
df_difference_statistics[df_difference_statistics['model_id'] == 2].groupby(['period'])['r2_difference'].describe().round(3)

,count,mean,std,min,25%,50%,75%,max


In [6]:
df_difference_statistics[df_difference_statistics['model_id'] == 3].groupby(['period'])['r2_difference'].describe().round(3)

,count,mean,std,min,25%,50%,75%,max


In [7]:
df_difference_coefficients['coefficient_relative_difference_absolute'] = df_difference_coefficients['coefficient_relative_difference'].abs()

In [8]:
df_difference_coefficients.groupby(['model_id','period','removed_variable'])[['coefficient_relative_difference_absolute']].mean().round(3).reset_index().sort_values(by=['model_id','period','coefficient_relative_difference_absolute'], ascending=[True, True, False])

,model_id,period,removed_variable,coefficient_relative_difference_absolute
1,11,2020,density_median_effectively_domiciled_area,0.713
12,11,2020,percentage_workers_services,0.485
9,11,2020,percentage_votes_for_bolsonaro,0.337
7,11,2020,percentage_self_employed_workers,0.262
3,11,2020,gini,0.226
...,...,...,...,...
60,11,2022,percentage_estimated_households_in_informal_se...,0.078
61,11,2022,percentage_hospitalizations_diseases_inadequat...,0.058
69,11,2022,unemployment_rate,0.028
62,11,2022,percentage_indigenous_population,0.022


In [9]:
df_difference_coefficients.groupby(['model_id','period','removed_variable'])[['coefficient_relative_difference_absolute']].median().round(3).reset_index().sort_values(by=['model_id','period','coefficient_relative_difference_absolute'], ascending=[True, True, False])

,model_id,period,removed_variable,coefficient_relative_difference_absolute
1,11,2020,density_median_effectively_domiciled_area,0.252
9,11,2020,percentage_votes_for_bolsonaro,0.220
12,11,2020,percentage_workers_services,0.208
2,11,2020,expected_years_of_schooling_at_age_18,0.146
11,11,2020,percentage_workers_industry,0.144
...,...,...,...,...
61,11,2022,percentage_hospitalizations_diseases_inadequat...,0.046
60,11,2022,percentage_estimated_households_in_informal_se...,0.014
69,11,2022,unemployment_rate,0.014
62,11,2022,percentage_indigenous_population,0.011


## Parameter sensitivity analysis

In [10]:
df_difference_statistics = pd.DataFrame()
df_difference_coefficients = pd.DataFrame()

list_periods = ['2020_1','2020', '2021', '2022', '2020_2022']

for model_id in [6,11]:
    for i in range(5):
        period = list_periods[i]
        
        print('\n*** Period: ', period)
        print('\*** Model',model_id)
        print('===>Full model:')
        with open('models/model_'+str(model_id)+'_'+period+'.pkl', 'rb') as file:
            model_reference = pickle.load(file)        
        # summarize_results(model_reference)
        model_reference_r2_cs = model_reference.pseudo_rsquared('cs')
        model_reference_r2_mcf = model_reference.pseudo_rsquared('mcf')
        model_reference_llf = model_reference.llf
        model_reference_aic = model_reference.aic
        model_reference_bic = model_reference.bic_llf
                
        for sample in range(30):
            print('\n*** Sample: ', sample)        
            with open('models/sensitivity_analysis/parameter/model_' + str(model_id) +'_' + period +'_sample_' + str(sample) + '.pkl', 'rb') as file:
                model_sample = pickle.load(file)        
            # summarize_results(model_variable)
            model_sample_r2_cs = model_sample.pseudo_rsquared('cs')
            model_sample_r2_mcf = model_sample.pseudo_rsquared('mcf')
            model_sample_llf = model_sample.llf
            model_sample_aic = model_sample.aic
            model_sample_bic = model_sample.bic_llf
            
            r2_cs_difference = model_sample_r2_cs - model_reference_r2_cs
            r2_mcf_difference = model_sample_r2_mcf - model_reference_r2_mcf
            llf_difference = model_sample_llf - model_reference_llf
            aic_difference = model_sample_aic - model_reference_aic
            bic_difference = model_sample_bic - model_reference_bic
            statistic_row = {'model_id':model_id, 'period': period, 'sample':sample, 'r2_cs_difference':r2_cs_difference, 'r2_mcf_difference': r2_mcf_difference, 'llf_difference': llf_difference, 'aic_difference': aic_difference, 'bic_difference': bic_difference}
            new_row = pd.DataFrame([statistic_row])
            df_difference_statistics = pd.concat([df_difference_statistics, new_row], ignore_index=True)
            
            for variable in model_sample.params.index:
                coefficient_difference = model_sample.params[variable] - model_reference.params[variable]
                coefficient_relative_difference = coefficient_difference / model_reference.params[variable]
                rate_ratio_difference = np.exp(model_sample.params[variable]) - np.exp(model_reference.params[variable])
                coefficient_row = {'model_id': model_id, 'period': period, 'sample': sample, 'variable': variable, 'coefficient_difference': coefficient_difference, 'coefficient_relative_difference': coefficient_relative_difference, 'rate_ratio_difference': rate_ratio_difference}
                new_row = pd.DataFrame([coefficient_row])
                df_difference_coefficients = pd.concat([df_difference_coefficients, new_row], ignore_index=True)


*** Period:  2020_1
\*** Model 6
===>Full model:

*** Sample:  0

*** Sample:  1

*** Sample:  2

*** Sample:  3

*** Sample:  4

*** Sample:  5

*** Sample:  6

*** Sample:  7

*** Sample:  8

*** Sample:  9

*** Sample:  10

*** Sample:  11

*** Sample:  12

*** Sample:  13

*** Sample:  14

*** Sample:  15

*** Sample:  16

*** Sample:  17

*** Sample:  18

*** Sample:  19

*** Sample:  20

*** Sample:  21

*** Sample:  22

*** Sample:  23

*** Sample:  24

*** Sample:  25

*** Sample:  26

*** Sample:  27

*** Sample:  28

*** Sample:  29

*** Period:  2020
\*** Model 6
===>Full model:

*** Sample:  0

*** Sample:  1

*** Sample:  2

*** Sample:  3

*** Sample:  4

*** Sample:  5

*** Sample:  6

*** Sample:  7

*** Sample:  8

*** Sample:  9

*** Sample:  10

*** Sample:  11

*** Sample:  12

*** Sample:  13

*** Sample:  14

*** Sample:  15

*** Sample:  16

*** Sample:  17

*** Sample:  18

*** Sample:  19

*** Sample:  20

*** Sample:  21

*** Sample:  22

*** Sample:  23

***

In [11]:
df_difference_statistics['r2_cs_difference_absolute'] = df_difference_statistics['r2_cs_difference'].abs()
df_difference_statistics.groupby(['model_id','period'])['r2_cs_difference'].describe().round(3)

count   mean    std    min    25%    50%    75%    max
model_id period                                                           
6        2020        30.0  0.002  0.011 -0.032 -0.005  0.001  0.009  0.023
         2020_1      30.0  0.002  0.020 -0.031 -0.013  0.002  0.016  0.048
         2020_2022   30.0  0.006  0.011 -0.017 -0.001  0.006  0.015  0.022
         2021        30.0 -0.000  0.013 -0.024 -0.009 -0.002  0.007  0.027
         2022        30.0  0.000  0.008 -0.015 -0.005 -0.001  0.005  0.023
11       2020        30.0  0.005  0.010 -0.021 -0.002  0.007  0.011  0.017
         2020_1      30.0  0.008  0.024 -0.036 -0.008  0.008  0.017  0.058
         2020_2022   30.0  0.014  0.013 -0.012  0.003  0.014  0.024  0.039
         2021        30.0 -0.005  0.015 -0.044 -0.015 -0.005  0.006  0.023
         2022        30.0  0.002  0.014 -0.025 -0.005  0.000  0.012  0.037

In [12]:
list_period_labels = ['2020 (first half)','2020','2021','2022','2020-2022']

In [13]:
final_summary_df = pd.DataFrame()
for statistic_difference in ['r2_cs_difference','r2_mcf_difference', 'llf_difference', 'aic_difference', 'bic_difference']:
    summary_df = df_difference_statistics.groupby(['model_id', 'period'])[statistic_difference].apply(calculate_95_ci).apply(pd.Series).reset_index()
    summary_df.columns = ['model_id', 'period', 'mean', 'lower_ci', 'upper_ci']
    
    summary_df['mean'] = summary_df['mean'].round(3)
    summary_df['lower_ci'] = summary_df['lower_ci'].round(3)
    summary_df['upper_ci'] = summary_df['upper_ci'].round(3)
    
    summary_df['CI'] = summary_df.apply(lambda row: f"{row['mean']}\n({row['lower_ci']}, {row['upper_ci']})", axis=1)
    
    summary_df_pivoted = summary_df[['model_id', 'period','CI']].sort_values('period').pivot(index='model_id', columns='period', values='CI')
    
    summary_df_pivoted = summary_df_pivoted.rename(columns={'2020_1': '2020 (first half)', '2020_2022': '2020-2022'})
    
    summary_df_pivoted = summary_df_pivoted[list_period_labels].reset_index(drop=False)
    summary_df_pivoted['statistic_difference'] = statistic_difference
    
    final_summary_df = pd.concat([final_summary_df, summary_df_pivoted], ignore_index=True)
final_summary_df = final_summary_df[['model_id','statistic_difference', '2020 (first half)', '2020', '2021', '2022', '2020-2022']]
final_summary_df.loc[final_summary_df['statistic_difference'] == 'r2_cs_difference', 'statistic_difference'] = '$R_{CS}^{2}$'
final_summary_df.loc[final_summary_df['statistic_difference'] == 'r2_mcf_difference', 'statistic_difference'] = '$R_{McF}^{2}$'
final_summary_df.loc[final_summary_df['statistic_difference'] == 'llf_difference', 'statistic_difference'] = 'LL'
final_summary_df.loc[final_summary_df['statistic_difference'] == 'aic_difference', 'statistic_difference'] = 'AIC'
final_summary_df.loc[final_summary_df['statistic_difference'] == 'bic_difference', 'statistic_difference'] = 'BIC'
final_summary_df.to_csv('data/df_sensitivity_analysis_bootstrap_statistic_difference.csv', index=False)

In [14]:
final_summary_df

period,model_id,statistic_difference,2020 (first half),2020,2021,2022,2020-2022
0,6,$R_{CS}^{2}$,"0.002\n(-0.006, 0.01)","0.002\n(-0.003, 0.006)","-0.0\n(-0.005, 0.005)","0.0\n(-0.003, 0.003)","0.006\n(0.002, 0.011)"
1,11,$R_{CS}^{2}$,"0.008\n(-0.001, 0.017)","0.005\n(0.001, 0.008)","-0.005\n(-0.011, 0.0)","0.002\n(-0.003, 0.007)","0.014\n(0.009, 0.019)"
2,6,$R_{McF}^{2}$,"0.0\n(-0.0, 0.001)","0.0\n(-0.0, 0.0)","-0.0\n(-0.0, 0.0)","0.0\n(-0.0, 0.0)","0.0\n(0.0, 0.001)"
3,11,$R_{McF}^{2}$,"0.0\n(-0.0, 0.001)","0.0\n(0.0, 0.0)","-0.0\n(-0.001, 0.0)","0.0\n(-0.0, 0.0)","0.001\n(0.001, 0.001)"
4,6,LL,"8.761\n(-17.117, 34.639)","10.834\n(-13.552, 35.22)","19.609\n(-1.004, 40.221)","15.312\n(-9.144, 39.767)","17.459\n(-4.249, 39.166)"
5,11,LL,"-0.451\n(-26.907, 26.004)","5.288\n(-16.268, 26.845)","-10.59\n(-30.931, 9.751)","23.983\n(-2.216, 50.182)","31.913\n(10.001, 53.824)"
6,6,AIC,"-17.521\n(-69.277, 34.235)","-21.668\n(-70.44, 27.104)","-39.217\n(-80.442, 2.007)","-30.624\n(-79.535, 18.288)","-34.917\n(-78.333, 8.498)"
7,11,AIC,"0.903\n(-52.008, 53.813)","-10.577\n(-53.691, 32.537)","21.18\n(-19.501, 61.862)","-47.967\n(-100.365, 4.432)","-63.825\n(-107.648, -20.003)"
8,6,BIC,"-17.521\n(-69.277, 34.235)","-21.668\n(-70.44, 27.104)","-39.217\n(-80.442, 2.007)","-30.624\n(-79.535, 18.288)","-34.917\n(-78.333, 8.498)"
9,11,BIC,"0.903\n(-52.008, 53.813)","-10.577\n(-53.691, 32.537)","21.18\n(-19.501, 61.862)","-47.967\n(-100.365, 4.432)","-63.825\n(-107.648, -20.003)"


In [15]:
df_difference_coefficients['coefficient_relative_difference_absolute'] = df_difference_coefficients['coefficient_relative_difference'].abs()

In [16]:
df_difference_coefficients.groupby(['model_id','period','variable'])[['coefficient_relative_difference_absolute']].mean().round(3).reset_index().sort_values(by=['model_id','period','coefficient_relative_difference_absolute'], ascending=[True, True, False])['variable'].unique()

array(['Semi-urbanized', 'Rural with high human development',
       'Rural with low human development',
       'Urbanized with informal settlements', 'const',
       'percentage_workers_commerce',
       'percentage_hospitalizations_diseases_inadequate_sanitation',
       'percentage_urban_population',
       'demographic_density_in_informal_settlements',
       'percentage_workers_industry', 'gini', 'unemployment_rate',
       'percentage_self_employed_workers',
       'percentage_indigenous_population', 'percentage_workers_services',
       'percentage_estimated_households_in_informal_settlements',
       'percentage_votes_for_bolsonaro',
       'expected_years_of_schooling_at_age_18',
       'density_median_effectively_domiciled_area'], dtype=object)

In [17]:
df_difference_coefficients['variable'].unique()

array(['const', 'Semi-urbanized', 'Rural with high human development',
       'Urbanized with informal settlements',
       'Rural with low human development', 'percentage_urban_population',
       'percentage_indigenous_population',
       'density_median_effectively_domiciled_area', 'gini',
       'percentage_estimated_households_in_informal_settlements',
       'demographic_density_in_informal_settlements',
       'percentage_hospitalizations_diseases_inadequate_sanitation',
       'percentage_self_employed_workers', 'unemployment_rate',
       'percentage_workers_commerce', 'percentage_workers_services',
       'percentage_workers_industry',
       'expected_years_of_schooling_at_age_18',
       'percentage_votes_for_bolsonaro'], dtype=object)

In [18]:
summary_df

,model_id,period,mean,lower_ci,upper_ci,CI
0,6,2020,-21.668,-70.440,27.104,"-21.668\n(-70.44, 27.104)"
1,6,2020_1,-17.521,-69.277,34.235,"-17.521\n(-69.277, 34.235)"
2,6,2020_2022,-34.917,-78.333,8.498,"-34.917\n(-78.333, 8.498)"
3,6,2021,-39.217,-80.442,2.007,"-39.217\n(-80.442, 2.007)"
4,6,2022,-30.624,-79.535,18.288,"-30.624\n(-79.535, 18.288)"
5,11,2020,-10.577,-53.691,32.537,"-10.577\n(-53.691, 32.537)"
6,11,2020_1,0.903,-52.008,53.813,"0.903\n(-52.008, 53.813)"
7,11,2020_2022,-63.825,-107.648,-20.003,"-63.825\n(-107.648, -20.003)"
8,11,2021,21.180,-19.501,61.862,"21.18\n(-19.501, 61.862)"
9,11,2022,-47.967,-100.365,4.432,"-47.967\n(-100.365, 4.432)"


In [19]:
summary_df = df_difference_coefficients[df_difference_coefficients['model_id']==11].groupby(['period','variable'])['coefficient_difference'].apply(calculate_95_ci).apply(pd.Series).reset_index()

summary_df.columns = ['period','variable', 'mean', 'lower_ci', 'upper_ci']

summary_df['mean'] = summary_df['mean'].apply(lambda x: f"{x:.3f}")
summary_df['lower_ci'] = summary_df['lower_ci'].apply(lambda x: f"{x:.3f}")
summary_df['upper_ci'] = summary_df['upper_ci'].apply(lambda x: f"{x:.3f}")

summary_df['CI'] = summary_df.apply(lambda row: f"{row['mean']}\n({row['lower_ci']}, {row['upper_ci']})", axis=1)

final_summary_df = summary_df[['period','variable','CI']].sort_values('period').pivot(index='variable', columns='period', values='CI')

final_summary_df = final_summary_df.rename(columns={'2020_1': '2020 (first half)', '2020_2022': '2020-2022'})

final_summary_df = final_summary_df[list_period_labels].reset_index()

final_summary_df = final_summary_df.set_index('variable')

final_summary_df = final_summary_df.loc[['const',
       'percentage_urban_population', 'density_median_effectively_domiciled_area',
       'percentage_indigenous_population', 'gini',
       'percentage_estimated_households_in_informal_settlements',
       'demographic_density_in_informal_settlements',
       'percentage_hospitalizations_diseases_inadequate_sanitation',
       'percentage_self_employed_workers', 'unemployment_rate',
       'percentage_workers_commerce', 'percentage_workers_services',
       'percentage_workers_industry', 'expected_years_of_schooling_at_age_18',
       'percentage_votes_for_bolsonaro']].copy()

final_summary_df.index = ['Intercept', '% urban population', 'Median density of effectively domiciled areas (inhabitants/km²)', '% indigenous population', 'Gini coefficient', '% informal settlement households', 'Population density in informal settlement (inhabitants/ha)', '% sanitation-related hospitalizations', '% self-employed workers', 'Unemployment rate', '% commerce workers', '% service workers', '% industry workers', 'Expected years of schooling at age 18','% votes for Bolsonaro']

final_summary_df.to_csv('data/df_sensitivity_analysis_bootstrap_coefficients_difference_model_11.csv', index=True)

In [20]:
summary_df = df_difference_coefficients[df_difference_coefficients['model_id']==6].groupby(['period','variable'])['coefficient_difference'].apply(calculate_95_ci).apply(pd.Series).reset_index()

summary_df.columns = ['period','variable', 'mean', 'lower_ci', 'upper_ci']

summary_df['mean'] = summary_df['mean'].apply(lambda x: f"{x:.3f}")
summary_df['lower_ci'] = summary_df['lower_ci'].apply(lambda x: f"{x:.3f}")
summary_df['upper_ci'] = summary_df['upper_ci'].apply(lambda x: f"{x:.3f}")

summary_df['CI'] = summary_df.apply(lambda row: f"{row['mean']}\n({row['lower_ci']}, {row['upper_ci']})", axis=1)

final_summary_df = summary_df[['period','variable','CI']].sort_values('period').pivot(index='variable', columns='period', values='CI')

final_summary_df = final_summary_df.rename(columns={'2020_1': '2020 (first half)', '2020_2022': '2020-2022'})

final_summary_df = final_summary_df[list_period_labels].reset_index()

final_summary_df = final_summary_df.set_index('variable')

final_summary_df = final_summary_df.loc[['const','Semi-urbanized', 'Rural with high human development', 'Urbanized with informal settlements', 'Rural with low human development']].copy()

final_summary_df.index = ['Intercept', 'Semi-urbanized', 'Rural with high human development', 'Urbanized with informal settlements', 'Rural with low human development']

final_summary_df.to_csv('data/df_sensitivity_analysis_bootstrap_coefficients_difference_model_6.csv', index=True)

In [21]:
summary_df = df_difference_coefficients[df_difference_coefficients['model_id']==6].groupby(['period','variable'])['rate_ratio_difference'].apply(calculate_95_ci).apply(pd.Series).reset_index()

In [22]:

summary_df.columns = ['period','variable', 'mean', 'lower_ci', 'upper_ci']

summary_df['mean'] = summary_df['mean'].round(3)
summary_df['lower_ci'] = summary_df['lower_ci'].round(3)
summary_df['upper_ci'] = summary_df['upper_ci'].round(3)

summary_df['CI'] = summary_df.apply(lambda row: f"{row['mean']}\n({row['lower_ci']}, {row['upper_ci']})", axis=1)

final_summary_df = summary_df[['period','variable','CI']].sort_values('period').pivot(index='variable', columns='period', values='CI')

final_summary_df = final_summary_df.rename(columns={'2020_1': '2020 (first half)', '2020_2022': '2020-2022'})

final_summary_df = final_summary_df[list_period_labels].reset_index()

final_summary_df = final_summary_df.set_index('variable')

In [23]:
final_summary_df

period,2020 (first half),2020,2021,2022,2020-2022
variable,,,,,
Rural with high human development,"0.016\n(-0.047, 0.079)","0.002\n(-0.011, 0.015)","-0.001\n(-0.006, 0.004)","0.002\n(-0.01, 0.015)","0.002\n(-0.003, 0.006)"
Rural with low human development,"-0.001\n(-0.173, 0.17)","0.021\n(-0.012, 0.054)","-0.0\n(-0.008, 0.008)","-0.0\n(-0.01, 0.01)","0.003\n(-0.006, 0.012)"
Semi-urbanized,"0.01\n(-0.035, 0.055)","0.001\n(-0.009, 0.01)","0.001\n(-0.003, 0.005)","0.001\n(-0.004, 0.005)","0.0\n(-0.003, 0.004)"
Urbanized with informal settlements,"0.011\n(-0.124, 0.147)","0.016\n(-0.017, 0.049)","-0.001\n(-0.01, 0.008)","-0.001\n(-0.009, 0.008)","0.003\n(-0.007, 0.012)"
const,"-60.087\n(-322.3, 202.125)","-338.437\n(-1113.509, 436.635)","-1059.175\n(-2914.922, 796.572)","-100.153\n(-563.143, 362.837)","-1017.281\n(-3340.332, 1305.77)"


In [24]:
final_summary_df = final_summary_df.loc[['const',
                                         'Urbanized with informal settlements',
                                         'Semi-urbanized',
                                         'Rural with high human development',
                                         'Rural with low human development']].copy()

final_summary_df.to_csv('data/df_sensitivity_analysis_bootstrap_rate_ratio_difference_model_6.csv', index=True)

## Outlier Sensitivity Analysis

### Model 6: Mortality rate ratio using 'Urbanized' as the reference group

In [26]:
df_analysis_cluster = pd.DataFrame()
df_analysis_cluster['Sociodemographic cluster'] = ['Urbanized','Urbanized with informal settlements','Semi-urbanized','Rural with high human development','Rural with low human development','DF','Residual DF','Deviance',"\chi^2",'$R_{CS}^{2}$','$R_{McF}^{2}$','LL','AIC', 'BIC']
df_analysis_cluster = df_analysis_cluster.set_index('Sociodemographic cluster')

list_periods = ['2020_1','2020', '2021', '2022', '2020_2022']
list_death_rate_columns = ['Death rate (1/2020)', 'Death rate (2020)', 'Death rate (2021)', 'Death rate (2022)', 'Death rate (accumulated period)']
list_period_labels = ['2020 (first half)','2020','2021','2022','2020-2022']

for i in range(len(list_periods)):
    period = list_periods[i]
    period_label = list_period_labels[i]
    death_rate_column = list_death_rate_columns[i]

    model = 6
    model_label = str(model)
    model_file = str(model)
    with open('models/sensitivity_analysis/outliers/model_'+model_file+'_'+period+'.pkl', 'rb') as file:
        model = pickle.load(file)

    # Extract the coefficients and standard errors
    params = model.params[1:5]
    conf = model.conf_int()[1:5]
    conf.columns = ['Lower CI', 'Upper CI']
    
    # Calculate the rate ratios and their confidence intervals
    rate_ratios = np.exp(params)
    conf['Lower CI'] = np.exp(conf['Lower CI'])
    conf['Upper CI'] = np.exp(conf['Upper CI'])
    
    # Combine into a single DataFrame
    rate_ratio_df = pd.DataFrame({
        'Rate Ratio': rate_ratios,
        'Lower CI': conf['Lower CI'],
        'Upper CI': conf['Upper CI']
    }) 
    
    rate_ratio_df.index = rate_ratio_df.index.astype(str).str.replace('cluster_label_','')
    
    rate_ratio_df = rate_ratio_df.round(2)
    
    column_rate_ratio = period_label+' - RR (95% CI) - Model '+model_label
    df_analysis_cluster[column_rate_ratio] = rate_ratio_df['Rate Ratio'].astype(str) + '\n' + '('+rate_ratio_df['Lower CI'].astype(str) + '-'+rate_ratio_df['Upper CI'].astype(str) + ')'        
    
    df_analysis_cluster.loc['Residual DF', column_rate_ratio] = model.df_resid
    df_analysis_cluster.loc['Deviance', column_rate_ratio] = round(model.deviance,2)
    df_analysis_cluster.loc["\chi^2", column_rate_ratio] = round(model.pearson_chi2,2)
    df_analysis_cluster.loc['$R_{CS}^{2}$', column_rate_ratio] = round(model.pseudo_rsquared('cs'),2)
    df_analysis_cluster.loc['$R_{McF}^{2}$', column_rate_ratio] = round(model.pseudo_rsquared('mcf'),2)
    df_analysis_cluster.loc['$R_{McF}^{2}$', column_rate_ratio] = round(model.pseudo_rsquared('mcf'),2)
    df_analysis_cluster.loc['LL', column_rate_ratio] = int(round(model.llf,0))
    df_analysis_cluster.loc['AIC', column_rate_ratio] = int(round(model.aic,0))
    df_analysis_cluster.loc['BIC', column_rate_ratio] = int(round(model.bic_llf,0))
    df_analysis_cluster.loc['DF', column_rate_ratio] = round(model.df_model)

    df_analysis_cluster[column_rate_ratio] = df_analysis_cluster[column_rate_ratio].fillna('1 [Reference]')

df_analysis_cluster.to_csv('data/df_analysis_cluster_outlier_sensitivity_6.csv', index=True)

### Model 11: Variables

In [31]:
df_results = pd.DataFrame()
df_results['Variable'] = ['const',
       'percentage_urban_population', 'density_median_effectively_domiciled_area',
       'percentage_indigenous_population', 'gini',
       'percentage_estimated_households_in_informal_settlements',
       'demographic_density_in_informal_settlements',
       'percentage_hospitalizations_diseases_inadequate_sanitation',
       'percentage_self_employed_workers', 'unemployment_rate',
       'percentage_workers_commerce', 'percentage_workers_services',
       'percentage_workers_industry', 'expected_years_of_schooling_at_age_18',
       '% people fully vaccinated','percentage_votes_for_bolsonaro','DF','Residual DF','Deviance',"\chi^2",'$R_{CS}^{2}$','$R_{McF}^{2}$','LL','AIC', 'BIC']
df_results = df_results.set_index('Variable')

list_periods = ['2020_1','2020', '2021', '2022', '2020_2022']
list_period_labels = ['2020 (first half)','2020','2021','2022','2020-2022']

for i in range(len(list_periods)):
    period = list_periods[i]
    period_label = list_period_labels[i]

    model = 11
    model_label = str(model)
    model_file = str(model)
    with open('models/sensitivity_analysis/outliers/model_'+model_file+'_'+period+'.pkl', 'rb') as file:
        model = pickle.load(file)

    # Extract the coefficients and standard errors
    params = model.params[:]
    conf = model.conf_int()[:]
    conf.columns = ['Lower CI', 'Upper CI']

    # Calculate the rate ratios and their confidence intervals
    rate_ratios = np.exp(params)
    conf['Lower CI'] = np.exp(conf['Lower CI'])
    conf['Upper CI'] = np.exp(conf['Upper CI'])

    # Combine into a single DataFrame
    rate_ratio_df = pd.DataFrame({
        'Rate Ratio': rate_ratios,
        'Lower CI': conf['Lower CI'],
        'Upper CI': conf['Upper CI']
    })

    rate_ratio_df.index = rate_ratio_df.index.astype(str).str.replace('cluster_label_','')

    rate_ratio_df = rate_ratio_df.round(2)

    column_rate_ratio = period_label+' - RR (95% CI) - Model '+model_label
    df_results[column_rate_ratio] = rate_ratio_df['Rate Ratio'].astype(str) + '\n' + '('+rate_ratio_df['Lower CI'].astype(str) + '-'+rate_ratio_df['Upper CI'].astype(str) + ')'

    df_results.loc['Residual DF', column_rate_ratio] = model.df_resid
    df_results.loc['Deviance', column_rate_ratio] = round(model.deviance,2)
    df_results.loc["\chi^2", column_rate_ratio] = round(model.pearson_chi2,2)
    df_results.loc['$R_{CS}^{2}$', column_rate_ratio] = round(model.pseudo_rsquared('cs'),2)
    df_results.loc['$R_{McF}^{2}$', column_rate_ratio] = round(model.pseudo_rsquared('mcf'),2)
    df_results.loc['$R_{McF}^{2}$', column_rate_ratio] = round(model.pseudo_rsquared('mcf'),2)
    df_results.loc['LL', column_rate_ratio] = int(round(model.llf,0))
    df_results.loc['AIC', column_rate_ratio] = int(round(model.aic,0))
    df_results.loc['BIC', column_rate_ratio] = int(round(model.bic_llf,0))
    df_results.loc['DF', column_rate_ratio] = round(model.df_model)

df_results.index = ['Intercept', '% urban population', 'Median density of effectively domiciled areas (inhabitants/km²)', '% indigenous population', 'Gini coefficient', '% informal settlement households', 'Population density in informal settlement (inhabitants/ha)', '% sanitation-related hospitalizations', '% self-employed workers', 'Unemployment rate', '% commerce workers', '% service workers', '% industry workers', 'Expected years of schooling at age 18','% people fully vaccinated','% votes for Bbolsonaro','DF','Residual DF','Deviance',"\chi^2",'$R_{CS}^{2}$','$R_{McF}^{2}$','LL','AIC', 'BIC']

df_results.to_csv('data/df_analysis_model_11_sensitivity_analysis.csv', index=True)